In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from pathlib import Path

notebook_path = "/u/skarmakar1/version_check/llm_steering-main/sk"
sys.path.append(notebook_path)

In [3]:
import torch
import numpy as np

from inversion_utils import *
import pickle
from sklearn.model_selection import train_test_split

In [4]:
SEED = 0
# SEED = 1

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

torch.backends.cudnn.benchmark = True 
torch.backends.cuda.matmul.allow_tf32 = True

LLM = namedtuple('LLM', ['language_model', 'tokenizer', 'processor', 'name', 'model_type'])

In [5]:
import random
import json

# random.seed(SEED)

In [6]:
model_type = 'llama'
MODEL_VERSION='3.1'
MODEL_SIZE='8B'

# model_type = 'gemma'
# MODEL_VERSION='2'
# # MODEL_VERSION='3'
# # MODEL_SIZE='1B'
# MODEL_SIZE='9B'
# # MODEL_SIZE='12B'

# model_type = 'qwen'
# MODEL_VERSION='3'
# MODEL_SIZE='4B'
# # MODEL_SIZE='8B'

llm = select_llm(model_type, MODEL_VERSION=MODEL_VERSION, MODEL_SIZE=MODEL_SIZE)

Loading meta-llama/Meta-Llama-3.1-8B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [7]:
hidden_layers = list(range(-llm.language_model.config.num_hidden_layers+1, 0))
print(hidden_layers)

[-31, -30, -29, -28, -27, -26, -25, -24, -23, -22, -21, -20, -19, -18, -17, -16, -15, -14, -13, -12, -11, -10, -9, -8, -7, -6, -5, -4, -3, -2, -1]


In [ ]:
# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/combined_models.pkl', 'rb') as file:
# # with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/combined_all_layers_models.pkl', 'rb') as file:
#     lrr_models = pickle.load(file)

In [ ]:
# categories = ["physical", "texture", "time", "complexity", "logic", "state", "social"]
# lrr_models = {}

# for category in categories:
#     with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/{category}_models.pkl', 'rb') as file:
#         lrr_models[category] = pickle.load(file)

In [8]:
test_set = [('itchy', 'relieved', 'texture'), ('colorful', 'colorless', 'texture'), ('expansive', 'confined', 'physical'), ('fat', 'skinny', 'physical'), ('sympathetic', 'unsympathetic', 'social'), ('kind', 'cruel', 'social'), ('dynamic', 'stagnant', 'time'), ('just', 'unjust', 'social'), ('clever', 'clumsy', 'logic'), ('towering', 'squat', 'physical'), ('perpendicular', 'parallel', 'physical'), ('precise', 'inaccurate', 'complexity'), ('solid', 'hollow', 'physical'), ('rotten', 'ripe', 'texture'), ('agonizing', 'soothing', 'texture'), ('squeaky', 'deep', 'texture'), ('valid', 'void', 'state'), ('incompetent', 'competent', 'complexity'), ('friendly', 'unfriendly', 'social'), ('clean', 'dirty', 'physical'), ('continuous', 'sporadic', 'time'), ('polite', 'rude', 'social'), ('virgin', 'used', 'time'), ('scented', 'odorless', 'texture'), ('amateurish', 'professional', 'complexity'), ('knotty', 'smooth', 'complexity'), ('early', 'late', 'time'), ('developing', 'declining', 'time'), ('brave', 'cowardly', 'social'), ('right', 'wrong', 'logic'), ('huge', 'tiny', 'physical'), ('good', 'evil', 'social'), ('serene', 'agitated', 'social'), ('harmonious', 'grating', 'texture'), ('sure', 'uncertain', 'complexity'), ('tangible', 'intangible', 'logic'), ('righteous', 'wicked', 'social'), ('here', 'gone', 'state'), ('soaked', 'parched', 'physical'), ('strenuous', 'facile', 'complexity'), ('diurnal', 'nocturnal', 'time'), ('wide', 'narrow', 'physical'), ('long-term', 'short-term', 'time'), ('obligatory', 'discretionary', 'logic'), ('hungry', 'sated', 'state'), ('innocent', 'guilty', 'social'), ('deep', 'superficial', 'complexity'), ('specific', 'general', 'complexity'), ('energetic', 'weary', 'state'), ('swift', 'leisurely', 'physical'), ('safe', 'dangerous', 'state'), ('unblemished', 'marred', 'state'), ('required', 'optional', 'logic'), ('activated', 'deactivated', 'state'), ('obscured', 'revealed', 'state'), ('arduous', 'effortless', 'complexity'), ('objective', 'subjective', 'logic'), ('playful', 'somber', 'social'), ('legal', 'illegal', 'logic'), ('logical', 'illogical', 'logic'), ('fugitive', 'durable', 'time'), ('manageable', 'unwieldy', 'complexity'), ('magnanimous', 'petty', 'social'), ('troublesome', 'convenient', 'complexity'), ('applicable', 'inapplicable', 'logic'), ('recent', 'remote', 'time'), ('dead', 'alive', 'state'), ('correct', 'incorrect', 'logic'), ('definite', 'vague', 'complexity'), ('chronic', 'acute', 'time'), ('animate', 'inanimate', 'logic'), ('covered', 'uncovered', 'state'), ('confident', 'insecure', 'social'), ('content', 'dissatisfied', 'social')]

In [9]:
base_prompts = {"complexity": 'Describe the following task, concept, or system. \nTopic: {statement}',
                "logic":'Analyze the following assertion. \nAssertion: {statement}', 
                "physical":'Write a description of the following item. \nItem: {statement}',
                "social":'What are your thoughts on the following statement? \nStatement: {statement}',
                "state":'Describe the current state of the following item. \nItem: {statement}',
                "texture":'Describe the sensory experience of interacting with the following item. \nItem: {statement}',
                "time":'Describe the following event or entity. \nSubject: {statement}',}

In [10]:
categories = ["physical", "texture", "time", "complexity", "logic", "state", "social"]
all_statements = {}

for category in categories:
    all_statements[category] = read_to_list(f"../data/general_statements_adj/{category}/test.txt")

In [11]:
def generate_eval_data(test_set, lrr_models=None, coef=0.75, max_tokens=100, n=5):
    send_to_eval = {}
    i = 0

    for concept in test_set:
        category = concept[2]
        c = concept[0]
        c_inv = concept[1]

        category_statements = all_statements[category]
        
        rand_statements = random.sample(category_statements, n)


        prompts = [base_prompts[category].format(statement=s) for s in rand_statements]

        c_controller = load_controller(llm, c, path=f'../all_gitignore/directions_adjectives_llama/{category}/')

        orig_out = test_concept_vector(c_controller, concept=c, prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

        if lrr_models is None:
            print(f"Using {c_inv} steering vector.")
            c_controller = load_controller(llm, c_inv, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
        elif "physical" in lrr_models:
            print(f"Using the {category} Inversion Matrix.")
            inv_c = apply_auto(c_controller.directions, lrr_models[category])
            c_controller.directions = inv_c
        else:
            print("Using the combined Inversion Matrix.")
            inv_c = apply_auto(c_controller.directions, lrr_models)
            c_controller.directions = inv_c
        
        pred_out = test_concept_vector(c_controller, concept=f"inv {c}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

        for o in range(len(orig_out)):
            u = clean_llama(orig_out[o]).find("-----------------------------------------------------")
            inv_u = clean_llama(orig_out[o]).find("-----------------------------------------------------")

            send_to_eval[i] = {"Target Concept" : c,
                                "Antonym": c_inv,
                                "Category": category,
                                "User Prompt": prompts[o],
                                "Original Response": clean_llama(orig_out[o])[u+54:],
                                "Predicted Antonym Response": clean_llama(pred_out[o])[inv_u+54:],}

            i += 1
            
        # break
    
    return send_to_eval

In [12]:
send_to_eval = generate_eval_data(test_set, n=5)
# send_to_eval = generate_eval_data(test_set, lrr_models=lrr_models, n=5)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found

========================== + itchy Control (normal) ==========================
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Describe the sensory experience of interacting with the following item. 
Item: The fluff of the dandelion.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The soft, gentle brush of your fingers against the delicate, fuzzy surface of the dandelion's seed head. The soft, almost whispery sound of the seeds brushing against each other as yo

In [13]:
# filename = "../all_gitignore/eval_json_auto/inv/combined.json"
# filename = "../all_gitignore/eval_json_auto/inv/separate.json"
filename = "../all_gitignore/eval_json_auto/inv/orig.json"

with open(filename, 'w') as f:
    json.dump(send_to_eval, f, indent=4)

Parser

In [ ]:
# files = ["1_response.json", "2_response.json", "3_response.json", "4_response.json"]

# all_res = {}

# for f in files:
#     # with open(f'eval_json/original/{f}', 'r', encoding='utf-8') as file:
#     with open(f'eval_json/inv_matrix_separate/{f}', 'r', encoding='utf-8') as file:
#     # with open(f'eval_json/inv_matrix/{f}', 'r', encoding='utf-8') as file:
#     # with open(f'eval_json/inv_matrix_all_layer/{f}', 'r', encoding='utf-8') as file:
#         data = json.load(file)
    
#     all_res.update(data)

# print(len(all_res))

740


In [ ]:
# score = 0

# for ind in all_res:
#     score += all_res[ind]['Evaluation Score']

# score = score / len(all_res)
# print(score)

6.981081081081081


In [ ]:
# Original steering vectors:                            7.943243243243243, 7.77027027027027
# With the inversion matrix (separate):                 6.981081081081081
# With the inversion matrix (combined):                 7.316216216216216
# With the inversion matrix (combined + all layers) :   7.108108108108108